# DeepFM on Amazon Books — Sparse Feature Interactions

End-to-end demo of `DeepFMClassifier` on the Amazon Reviews 2023 — Books category.

**Why DeepFM here?** DeepFM combines a factorization-machine wide arm (which learns
second-order feature interactions) with a deep MLP arm (which learns higher-order
non-linearities). It shines when there are many sparse categorical features, and that's
exactly what Amazon Books gives us — categories, publishers, price tiers, popularity bins.
MovieLens is too low-cardinality to make DeepFM stand out; Books is the right showcase.

**Comparison:** numbers should be reported alongside the LightGBM baseline from
[`lightgbm_amazon_books.ipynb`](lightgbm_amazon_books.ipynb) for a tabular-vs-DeepFM
contrast on the same data + protocol.

**Evaluation protocol**: leave-last-positive-out, sampled ranking (1 positive + 100 negatives).

## 1. Imports

In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.estimator.classification.deep_fm_classifier import DeepFMClassifier
from skrec.recommender.ranking.ranking_recommender import RankingRecommender
from skrec.scorer.universal import UniversalScorer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/deepfm")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

Imports OK


## 2. Download + sample Amazon Books

We pull the McAuley Lab **Amazon Reviews 2023** 5-core Books rating CSV directly from
HuggingFace via `hf_hub_download` (one ~525 MB file, no `datasets` library needed). This
gives us `(user, item, rating, timestamp)` for every 5-core review. We then sample 100k
users with a deterministic seed.

The full Books metadata file (titles, categories, publisher, price) is 14 GB — impractical
for a notebook. These notebooks therefore rely on **interaction-derived features only**:
item popularity, item rating statistics, user behavioural statistics. The recommender
displays book IDs in place of titles.

The cached parquet is **shared across all four Books notebooks** — first run pays the
download cost (~1–3 min); the rest hit the cache instantly.

In [2]:
INTERACTIONS_PARQUET = RAW_DIR / "interactions.parquet"

TARGET_N_USERS = 100_000
SEED = 42

if INTERACTIONS_PARQUET.exists():
    print(f"Cache hit at {INTERACTIONS_PARQUET} — skipping download.")
else:
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        print("Installing huggingface_hub...")
        import subprocess
        import sys

        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import hf_hub_download

    print("Downloading 5core/rating_only/Books.csv (~525 MB)...")
    csv_path = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023",
        filename="benchmark/5core/rating_only/Books.csv",
        repo_type="dataset",
    )
    print(f"  -> {csv_path}")

    # The 5core rating-only CSV has columns: user_id, parent_asin, rating, timestamp.
    # The full file is ~9.5M rows; loading it all at default pandas dtypes can OOM.
    # Two-pass chunked approach instead:
    #   Pass 1 — stream user_id only, collect unique users, sample.
    #   Pass 2 — stream all 4 columns, keep only sampled users.
    CHUNK = 2_000_000
    print("Pass 1: streaming unique users...")
    unique_users: set[str] = set()
    for chunk in pd.read_csv(csv_path, chunksize=CHUNK, usecols=["user_id"], dtype={"user_id": str}):
        unique_users.update(chunk["user_id"].unique())
    print(f"  total unique users: {len(unique_users):,}")

    rng = np.random.default_rng(SEED)
    sampled_users = set(
        rng.choice(
            np.array(sorted(unique_users)),
            size=min(TARGET_N_USERS, len(unique_users)),
            replace=False,
        )
    )
    print(f"  sampled: {len(sampled_users):,}")

    print("Pass 2: streaming and filtering...")
    parts = []
    for chunk in pd.read_csv(
        csv_path,
        chunksize=CHUNK,
        dtype={"user_id": str, "parent_asin": str, "rating": "float32", "timestamp": "int64"},
    ):
        parts.append(chunk[chunk["user_id"].isin(sampled_users)])
    df = pd.concat(parts, ignore_index=True)
    df = df.rename(columns={"user_id": "USER_ID", "parent_asin": "ITEM_ID", "timestamp": "TIMESTAMP"})
    n_users = df["USER_ID"].nunique()
    n_items = df["ITEM_ID"].nunique()
    print(f"  kept: {len(df):,} interactions across {n_users:,} users, {n_items:,} items")

    df.to_parquet(INTERACTIONS_PARQUET)
    print(f"Saved -> {INTERACTIONS_PARQUET}")

interactions = pd.read_parquet(INTERACTIONS_PARQUET)
print(f"\nLoaded {len(interactions):,} interactions.")
print(f"  users: {interactions['USER_ID'].nunique():,}  items: {interactions['ITEM_ID'].nunique():,}")
interactions.head(3)

Cache hit at data/raw/interactions.parquet — skipping download.

Loaded 1,216,565 interactions.


  users: 100,000  items: 361,673


,USER_ID,ITEM_ID,rating,TIMESTAMP
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1446304000,5.0,1441260345000
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1564770672,5.0,1441260365000
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1442450703,5.0,1523093714024


## 3. Sparse-Feature Engineering (interaction-derived only)

The McAuley Books metadata file is 14 GB — too large to download for a notebook. We
instead derive sparse features from interactions and quantile-bin them so DeepFM's FM arm
sees genuinely sparse one-hot tiers it can cross:

- **Item popularity tier**: 5-bin quantile bucketing of log-review-count → 5 binary columns
- **Item rating-quality tier**: 5-bin quantile bucketing of mean rating → 5 binary columns
- **Item rating-variance tier**: 5-bin quantile bucketing of rating std → 5 binary columns

These will be merged with **user behavioural features** in the next cell. The FM arm
learns pairwise interactions like *"high-popularity tier × high-variance user"*
automatically. Without metadata categories, this notebook has fewer features than the
LightGBM-on-MovieLens reference, but the DeepFM architecture and pipeline are identical.

In [3]:
all_items = sorted(interactions["ITEM_ID"].unique())
items_df = pd.DataFrame({"ITEM_ID": all_items})
print(f"Catalog: {len(items_df):,} unique items")
items_df.head(3)

Catalog: 361,673 unique items


,ITEM_ID
0,0000013714
1,000100039X
2,0001061240


## 4. Train / Test Split (Leave-Last-Positive-Out)

Same protocol as the LightGBM notebook so the two models are directly comparable on
the same ranking task.

In [4]:
df = interactions.copy()
df["OUTCOME"] = (df["rating"] >= 4).astype(float)
df = df.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

counts = df.groupby("USER_ID").size()
df = df[df["USER_ID"].isin(counts[counts >= 3].index)].reset_index(drop=True)

pos = df[df["OUTCOME"] == 1.0]
last_pos_idx = pos.groupby("USER_ID")["TIMESTAMP"].idxmax()
test_df = pos.loc[last_pos_idx].reset_index(drop=True)
train_df = df.drop(index=last_pos_idx).reset_index(drop=True)
test_df = test_df[test_df["USER_ID"].isin(train_df["USER_ID"].unique())].reset_index(drop=True)

used_items = set(train_df["ITEM_ID"]).union(set(test_df["ITEM_ID"]))
items_df = items_df[items_df["ITEM_ID"].isin(used_items)].reset_index(drop=True)

print(f"Train : {len(train_df):,}  positive rate: {train_df.OUTCOME.mean():.1%}")
print(f"Test  : {len(test_df):,}  one per user")
print(f"Users : {train_df.USER_ID.nunique():,}  Items: {len(items_df):,}")

Train : 1,116,932  positive rate: 83.3%
Test  : 99,633  one per user
Users : 100,000  Items: 361,673


## 5. Compute item tiers + user features (from training data only)

Item-side: 5-bin quantile tiers for popularity, mean rating, and rating std.
User-side: same three behavioural features as the LightGBM notebook.

In [5]:
# --- item tiers ---
item_stats = (
    train_df.groupby("ITEM_ID")["rating"]
    .agg(item_rating_mean="mean", item_rating_count="count", item_rating_std="std")
    .fillna(0.0)
    .reset_index()
)
item_stats["item_rating_count_log"] = np.log1p(item_stats["item_rating_count"])


def _quantile_onehot(series, prefix, q=5):
    tiers = pd.qcut(series, q=q, labels=False, duplicates="drop")
    cols = {}
    for t in sorted(int(x) for x in tiers.dropna().unique()):
        cols[f"{prefix}_{t}"] = (tiers == t).astype(float).values
    return pd.DataFrame(cols, index=series.index)


pop_oh = _quantile_onehot(item_stats["item_rating_count_log"], "pop_tier")
qual_oh = _quantile_onehot(item_stats["item_rating_mean"], "qual_tier")
var_oh = _quantile_onehot(item_stats["item_rating_std"], "var_tier")

item_features = pd.concat([item_stats[["ITEM_ID"]], pop_oh, qual_oh, var_oh], axis=1)
items_df = items_df.merge(item_features, on="ITEM_ID", how="left").fillna(0.0)
print(f"Items × features: {items_df.shape}")

# --- user features ---
user_stats = (
    train_df.groupby("USER_ID")["rating"]
    .agg(user_rating_mean="mean", user_rating_count="count", user_rating_std="std")
    .fillna(0.0)
    .reset_index()
)
user_stats["user_rating_count_log"] = np.log1p(user_stats["user_rating_count"])
user_stats = user_stats[["USER_ID", "user_rating_mean", "user_rating_count_log", "user_rating_std"]]

train_df = train_df.merge(user_stats, on="USER_ID", how="left")
user_stats.head(3)

Items × features: (361673, 9)


,USER_ID,user_rating_mean,user_rating_count_log,user_rating_std
0,AE22236AFRRSMQIKGG7TPTB75QEA,4.615385,2.639057,0.767948
1,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,4.826087,3.178054,0.491026
2,AE226IAGN236VQPS5WLIWAXQAR5A,4.800000,1.791759,0.447214


## 6. Save CSVs and Build Datasets

In [6]:
train_path = str(DATA_DIR / "train_interactions.csv")
items_path = str(DATA_DIR / "items.csv")

train_to_save = train_df[
    ["USER_ID", "ITEM_ID", "OUTCOME", "user_rating_mean", "user_rating_count_log", "user_rating_std"]
]
if not Path(train_path).exists():
    train_to_save.to_csv(train_path, index=False)
if not Path(items_path).exists():
    items_df.to_csv(items_path, index=False)

interactions_ds = InteractionsDataset(data_location=train_path)
items_ds = ItemsDataset(data_location=items_path)
n_train, n_train_cols = len(train_to_save), train_to_save.shape[1]
n_items_total, n_item_cols = len(items_df), items_df.shape[1] - 1
print(f"Training: {n_train:,} × {n_train_cols}  Items: {n_items_total:,} × {n_item_cols}")

Training: 1,116,932 × 6  Items: 361,673 × 8


## 7. Build and Train DeepFM

Hyperparameters chosen to balance fit and runtime on a CPU:

- `embedding_dim=16` for the FM second-order arm
- 2-layer MLP `hidden_dim1=128, hidden_dim2=64` for higher-order interactions
- `use_cross_layer=True` adds a 2-layer cross network — DCN-style explicit feature crossing
  on top of the FM/MLP combo, useful when feature interactions are dense
- 5 epochs is plenty given how strong the gradient signal is on tabular data

Training takes ~3–5 minutes on CPU at this scale.

In [7]:
estimator = DeepFMClassifier(
    params={
        "embedding_dim": 16,
        "hidden_dim1": 128,
        "hidden_dim2": 64,
        "batch_size": 4096,
        "epochs": 5,
        "lr": 1e-3,
        "dropout": 0.1,
        "use_cross_layer": True,
        "num_cross_layers": 2,
        "use_batch_norm": True,
    }
)

scorer = UniversalScorer(estimator)
recommender = RankingRecommender(scorer)

print("Training DeepFM...")
recommender.train(interactions_ds=interactions_ds, items_ds=items_ds)
print("Done.")

Training DeepFM...


2026-04-29 00:32:12,275 - skrec.estimator.classification.deep_fm_classifier - INFO Epoch [1/5] - Train Loss: 0.3087


2026-04-29 00:32:12,275 skrec.estimator.classification.deep_fm_classifier INFO Epoch [1/5] - Train Loss: 0.3087


2026-04-29 00:32:14,266 - skrec.estimator.classification.deep_fm_classifier - INFO Epoch [2/5] - Train Loss: 0.2423


2026-04-29 00:32:14,266 skrec.estimator.classification.deep_fm_classifier INFO Epoch [2/5] - Train Loss: 0.2423


...

2026-04-29 00:32:18,126 - skrec.estimator.classification.deep_fm_classifier - INFO Epoch [4/5] - Train Loss: 0.2350


2026-04-29 00:32:18,126 skrec.estimator.classification.deep_fm_classifier INFO Epoch [4/5] - Train Loss: 0.2350


2026-04-29 00:32:19,984 - skrec.estimator.classification.deep_fm_classifier - INFO Epoch [5/5] - Train Loss: 0.2342


2026-04-29 00:32:19,984 skrec.estimator.classification.deep_fm_classifier INFO Epoch [5/5] - Train Loss: 0.2342


Done.


## 8. Evaluate: HR@10 and NDCG@10 (Sampled Ranking)

Same per-pair scoring approach as the LightGBM notebook — Books has too many items to
materialise a full score matrix. We score only `1 positive + 100 negatives` per user
on a 2,000-user random sample.

In [8]:
TOP_K, N_NEG, N_EVAL_USERS = 10, 100, 2000
rng = np.random.default_rng(42)

known_items = set(items_df["ITEM_ID"])
catalog_arr = np.array(items_df["ITEM_ID"])
eval_test = test_df[test_df["ITEM_ID"].isin(known_items)].copy()
if len(eval_test) > N_EVAL_USERS:
    eval_test = eval_test.sample(n=N_EVAL_USERS, random_state=42).reset_index(drop=True)
print(f"Evaluating {len(eval_test):,} users (sampled).")

train_seen = train_df.groupby("USER_ID")["ITEM_ID"].apply(set).to_dict()

pair_user, pair_item = [], []
gt = {}
for _, row in eval_test.iterrows():
    u = row["USER_ID"]
    test_item = row["ITEM_ID"]
    seen = train_seen.get(u, set())
    unseen = catalog_arr[~np.isin(catalog_arr, list(seen))]
    neg = rng.choice(unseen, size=min(N_NEG, len(unseen)), replace=False)
    pair_user.extend([u] * (1 + len(neg)))
    pair_item.append(test_item)
    pair_item.extend(neg)
    gt[u] = test_item

pairs = pd.DataFrame({"USER_ID": pair_user, "ITEM_ID": pair_item})
pairs = pairs.merge(user_stats, on="USER_ID", how="left")
pairs = pairs.merge(items_df, on="ITEM_ID", how="left").fillna(0.0)
X = pairs[estimator.feature_names]
print(f"Scoring {len(X):,} (user, candidate) pairs through DeepFM...")
scores = estimator.predict_proba(X)[:, 1]

n_users = len(eval_test)
n_per = 1 + N_NEG
scores_2d = scores.reshape(n_users, n_per)
test_scores = scores_2d[:, 0:1]
ranks = (scores_2d > test_scores).sum(axis=1) + 1
hit = (ranks <= TOP_K).astype(int)
ndcg = np.where(ranks <= TOP_K, 1.0 / np.log2(ranks + 1), 0.0)

print(f"\n{'=' * 40}")
print(f"Sampled ranking (1 pos + {N_NEG} neg)")
print(f"HR@{TOP_K}   : {hit.mean():.4f}")
print(f"NDCG@{TOP_K} : {ndcg.mean():.4f}")
print(f"Users : {n_users:,}")
print(f"{'=' * 40}")

eval_users_list = eval_test["USER_ID"].tolist()

Evaluating 2,000 users (sampled).


Scoring 202,000 (user, candidate) pairs through DeepFM...

Sampled ranking (1 pos + 100 neg)
HR@10   : 0.1555
NDCG@10 : 0.0930
Users : 2,000


## 9. Sample Recommendations

In [9]:
# No title metadata available — show truncated ITEM_ID instead.
for ui in range(5):
    u = eval_users_list[ui]
    user_pair_idx = slice(ui * n_per, (ui + 1) * n_per)
    user_items = pairs.iloc[user_pair_idx]["ITEM_ID"].values
    user_scores = scores[user_pair_idx]
    order = np.argsort(-user_scores)
    top = user_items[order][:TOP_K]
    test_item = gt.get(u, "?")
    flag = "HIT" if test_item in top else "MISS"
    print(f"\nUser {u}  |  Test: {test_item}  [{flag}]")
    for r, item_id in enumerate(top, 1):
        marker = "  <-- TEST" if item_id == test_item else ""
        print(f"  {r:2}. {item_id}{marker}")


User AE4YW7GY4LKJXP3LJKKYJAHJPOKA  |  Test: B00IB5BSBG  [HIT]
   1. 0500015643
   2. 1604688513
   3. 0345531973
   4. 0147518369
   5. B00IB5BSBG  <-- TEST
   6. B00ND4GF4K
   7. 044653630X
   8. 1595340416
   9. B06XCC6JJL
  10. 1770854754

User AF6LM4TAY5BOEGCP6LN4KQAMHI4Q  |  Test: 1574888862  [HIT]
   1. 1574888862  <-- TEST
   2. 1941235107
   3. 1631596152
   4. 0062002732
   5. 0877792917
   6. B01N5A4YMB
   7. 0873418069
   8. B07WH9J7J5
   9. 191008526X
  10. 1593691262

User AEEKCCQ73X6X6H65ZVPVMQHJEKYA  |  Test: 0380791714  [MISS]
   1. 0061579114
   2. 0345316509
   3. 034554983X
   4. B07Z2L6VP4
   5. 1984822179
   6. B01N2S9N0F
   7. 0060563443
   8. 0380732238
   9. 006208576X
  10. 110199018X

User AFTHGYDPP4RBKYNQHIK4MFG4RKLQ  |  Test: B01ATF489O  [MISS]
   1. 1933831057
   2. 0312313926
   3. 0793533279
   4. 1565126084
   5. B07TQJQSSB
   6. B004LQ0F5E
   7. 1631597639
   8. 1601404476
   9. 1596434600
  10. 1596356820

User AGF6MW22N7WP4UW3OUJBEIZCQ4FQ  |  Test: 1